<a href="https://colab.research.google.com/github/Pksingh1992/Short-Term-Household-Electricity-Forecasting/blob/main/Short_Term_Household_Electricity_Forecasting_A_Paper_Inspired_LSTM_Adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Abstract

This project studies short-term household electricity forecasting using hourly power consumption data. Inspired by a recent paper on large-scale electricity load forecasting, I build a single-household adaptation using temporal feature engineering and LSTM-based sequence modeling. The task is to predict the next 24 hours of electricity usage from the previous 192 hours. I evaluate the model with RMSE, MAE, MAPE, and sMAPE, and compare all-season forecasting with season-specific models. This project is not a full reproduction of the reference paper; instead, it is a practical end-to-end adaptation focused on one household.

## Problem Statement

The goal is to forecast the next 24 hours of household electricity demand using recent historical usage and calendar-based features. This is a multistep time-series regression problem with strong daily, weekly, and seasonal patterns.

## Relation to the Reference Paper

This project is inspired by the paper *Learning model combined with data clustering and dimensionality reduction for short-term electricity load forecasting*.

### Similarities
- short-term load forecasting setup
- log-transformed target
- cyclical encoding of time variables
- 192-hour input window and 24-hour prediction horizon
- seasonal analysis
- LSTM as a forecasting model

### Differences
- the paper studies 4710 households, while this project uses a single household
- the paper's key novelty is dimensionality reduction plus clustering before forecasting
- this project does not implement kernel PCA, UMAP, t-SNE, or k-means in the final pipeline
- this project focuses on LSTM and baseline forecasting comparisons

## Dataset

The dataset contains household electricity consumption measured over time. The raw data are resampled to hourly frequency so that the forecasting problem matches short-term hourly prediction.

## Preprocessing

The preprocessing pipeline includes:
- datetime parsing and sorting
- hourly resampling
- missing value handling
- log transformation of the target
- cyclical encoding of hour, day, and month
- lag features and rolling summary statistics

In [2]:
!pip install numpy pandas scikit-learn torch xgboost matplotlib tqdm holidays umap-learn

In [3]:
import zipfile
import os
import math
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import copy


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
base = '/content/drive/MyDrive'
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith('.zip'):
            print(os.path.join(root, f))

/content/drive/MyDrive/archive.zip
/content/drive/MyDrive/individual+household+electric+power+consumption (1).zip
/content/drive/MyDrive/python projects/project-optimization.zip


In [7]:
zip_path = '/content/drive/MyDrive/individual+household+electric+power+consumption (1).zip'
extract_path = '/content/data_folder'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('Unzip complete')
print(os.listdir(extract_path))

Unzip complete
['household_power_consumption.txt']


In [8]:
txt_path = '/content/data_folder/household_power_consumption.txt'
csv_path = '/content/data_folder/household_power_consumption.csv'

df = pd.read_csv(
    txt_path,
    sep=';',
    na_values='?',
    low_memory=False
)

df.to_csv(csv_path, index=False)

print('CSV saved to:', csv_path)
print(df.shape)
print(df.head())

CSV saved to: /content/data_folder/household_power_consumption.csv
(2075259, 9)
         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


In [9]:
df = pd.read_csv('/content/data_folder/household_power_consumption.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(2075259, 9)
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [10]:
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

df = df.sort_values('datetime').reset_index(drop=True)

print(df[['Date', 'Time', 'datetime']].head())
print(df[['Date', 'Time', 'datetime']].tail())

         Date      Time            datetime
0  16/12/2006  17:24:00 2006-12-16 17:24:00
1  16/12/2006  17:25:00 2006-12-16 17:25:00
2  16/12/2006  17:26:00 2006-12-16 17:26:00
3  16/12/2006  17:27:00 2006-12-16 17:27:00
4  16/12/2006  17:28:00 2006-12-16 17:28:00
               Date      Time            datetime
2075254  26/11/2010  20:58:00 2010-11-26 20:58:00
2075255  26/11/2010  20:59:00 2010-11-26 20:59:00
2075256  26/11/2010  21:00:00 2010-11-26 21:00:00
2075257  26/11/2010  21:01:00 2010-11-26 21:01:00
2075258  26/11/2010  21:02:00 2010-11-26 21:02:00


In [11]:
print(df['datetime'].min())
print(df['datetime'].max())
print(df.shape)

2006-12-16 17:24:00
2010-11-26 21:02:00
(2075259, 10)


In [12]:
use_cols = [
    'datetime',
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

data = df[use_cols].copy()

print(data.shape)
print(data.isna().sum())
data.head()

(2075259, 8)
datetime                     0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64


,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [13]:
data.dtypes

,0
datetime,datetime64[ns]
Global_active_power,float64
Global_reactive_power,float64
Voltage,float64
Global_intensity,float64
Sub_metering_1,float64
Sub_metering_2,float64
Sub_metering_3,float64


In [14]:
data = data.sort_values('datetime').reset_index(drop=True)

# time-based interpolation for numeric columns
num_cols = [c for c in data.columns if c != 'datetime']
data[num_cols] = data[num_cols].interpolate(method='linear', limit_direction='both')

print(data.isna().sum())
data.head()

datetime                 0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64


,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [15]:
hourly = data.set_index('datetime').resample('H').mean().reset_index()

print(hourly.shape)
hourly.head()
hourly.tail()

(34589, 8)


/tmp/ipykernel_2144/2941502269.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = data.set_index('datetime').resample('H').mean().reset_index()


,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
34584,2010-11-26 17:00:00,1.725900,0.061400,237.069667,7.216667,0.0,0.000000,12.866667
34585,2010-11-26 18:00:00,1.573467,0.053700,237.531833,6.620000,0.0,0.000000,0.000000
34586,2010-11-26 19:00:00,1.659333,0.060033,236.741000,7.056667,0.0,0.066667,0.000000
34587,2010-11-26 20:00:00,1.163700,0.061167,239.396000,4.913333,0.0,1.066667,0.000000
34588,2010-11-26 21:00:00,0.934667,0.000000,239.690000,3.800000,0.0,0.000000,0.000000


In [16]:
print(hourly.isna().sum())
print(hourly['datetime'].min())
print(hourly['datetime'].max())

datetime                 0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64
2006-12-16 17:00:00
2010-11-26 21:00:00


In [17]:
feat = hourly.copy()

# basic calendar parts
feat['hour'] = feat['datetime'].dt.hour
feat['day'] = feat['datetime'].dt.day
feat['month'] = feat['datetime'].dt.month
feat['day_of_week'] = feat['datetime'].dt.dayofweek   # Monday=0, Sunday=6
feat['days_in_month'] = feat['datetime'].dt.days_in_month

# cyclical encoding
feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)

feat['day_sin'] = np.sin(2 * np.pi * feat['day'] / feat['days_in_month'])
feat['day_cos'] = np.cos(2 * np.pi * feat['day'] / feat['days_in_month'])

feat['month_sin'] = np.sin(2 * np.pi * feat['month'] / 12)
feat['month_cos'] = np.cos(2 * np.pi * feat['month'] / 12)

# weekday one-hot columns
weekday_dummies = pd.get_dummies(feat['day_of_week'], prefix='wd')
feat = pd.concat([feat, weekday_dummies], axis=1)

print(feat.shape)
feat[['datetime', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos']].head()

(34589, 26)


,datetime,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,2006-12-16 17:00:00,-0.965926,-2.588190e-01,-0.101168,-0.994869,-2.449294e-16,1.0
1,2006-12-16 18:00:00,-1.000000,-1.836970e-16,-0.101168,-0.994869,-2.449294e-16,1.0
2,2006-12-16 19:00:00,-0.965926,2.588190e-01,-0.101168,-0.994869,-2.449294e-16,1.0
3,2006-12-16 20:00:00,-0.866025,5.000000e-01,-0.101168,-0.994869,-2.449294e-16,1.0
4,2006-12-16 21:00:00,-0.707107,7.071068e-01,-0.101168,-0.994869,-2.449294e-16,1.0


In [18]:
new_cols = [c for c in feat.columns if c not in hourly.columns]
print(new_cols)

['hour', 'day', 'month', 'day_of_week', 'days_in_month', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'wd_0', 'wd_1', 'wd_2', 'wd_3', 'wd_4', 'wd_5', 'wd_6']


In [19]:
feat2 = feat.copy()

# log transform of the main target
feat2['target'] = np.log1p(feat2['Global_active_power'])

# useful lag features from the target
feat2['lag_1'] = feat2['target'].shift(1)
feat2['lag_24'] = feat2['target'].shift(24)
feat2['lag_48'] = feat2['target'].shift(48)
feat2['lag_168'] = feat2['target'].shift(168)

# rolling features
feat2['roll_mean_24'] = feat2['target'].rolling(24).mean()
feat2['roll_std_24'] = feat2['target'].rolling(24).std()

feat2['roll_mean_168'] = feat2['target'].rolling(168).mean()
feat2['roll_std_168'] = feat2['target'].rolling(168).std()

print(feat2.shape)
feat2[['datetime', 'Global_active_power', 'target', 'lag_1', 'lag_24', 'lag_168']].head(30)

(34589, 35)


,datetime,Global_active_power,target,lag_1,lag_24,lag_168
0,2006-12-16 17:00:00,4.222889,1.653051,NaN,NaN,NaN
1,2006-12-16 18:00:00,3.632200,1.533032,1.653051,NaN,NaN
2,2006-12-16 19:00:00,3.400233,1.481658,1.533032,NaN,NaN
3,2006-12-16 20:00:00,3.268567,1.451278,1.481658,NaN,NaN
4,2006-12-16 21:00:00,3.056467,1.400312,1.451278,NaN,NaN
5,2006-12-16 22:00:00,2.200133,1.163192,1.400312,NaN,NaN
6,2006-12-16 23:00:00,2.061600,1.118938,1.163192,NaN,NaN
7,2006-12-17 00:00:00,1.882467,1.058646,1.118938,NaN,NaN
8,2006-12-17 01:00:00,3.349400,1.470038,1.058646,NaN,NaN
9,2006-12-17 02:00:00,1.587267,0.950602,1.470038,NaN,NaN


In [20]:
print(feat2.isna().sum().sort_values(ascending=False).head(15))

lag_168                  168
roll_std_168             167
roll_mean_168            167
lag_48                    48
lag_24                    24
roll_mean_24              23
roll_std_24               23
lag_1                      1
Global_reactive_power      0
datetime                   0
Global_active_power        0
Voltage                    0
Global_intensity           0
Sub_metering_1             0
Sub_metering_2             0
dtype: int64


In [21]:
model_df = feat2.dropna().reset_index(drop=True)

print(model_df.shape)
print(model_df.isna().sum().sum())
model_df.head()

(34421, 35)
0


,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,day,...,wd_6,target,lag_1,lag_24,lag_48,lag_168,roll_mean_24,roll_std_24,roll_mean_168,roll_std_168
0,2006-12-23 17:00:00,5.452533,0.215967,233.644167,23.360000,16.183333,0.666667,16.750000,17,23,...,False,1.864473,1.676928,0.915010,1.012558,1.653051,1.377426,0.267168,0.926138,0.442816
1,2006-12-23 18:00:00,3.879400,0.099767,238.000500,16.363333,0.000000,0.016667,17.350000,18,23,...,False,1.585022,1.864473,1.304804,1.236430,1.533032,1.389102,0.269965,0.926448,0.443261
2,2006-12-23 19:00:00,4.117833,0.205333,238.729333,17.300000,0.000000,0.600000,17.466667,19,23,...,False,1.632731,1.585022,1.596994,1.162255,1.481658,1.390591,0.271257,0.927347,0.444545
3,2006-12-23 20:00:00,4.181400,0.124767,238.518833,17.596667,0.000000,0.350000,17.416667,20,23,...,False,1.645075,1.632731,1.512060,1.234706,1.451278,1.396134,0.275177,0.928500,0.446161
4,2006-12-23 21:00:00,3.288433,0.235767,238.594667,13.893333,0.000000,0.216667,5.666667,21,23,...,False,1.455921,1.645075,1.713558,0.684241,1.400312,1.385399,0.267164,0.928831,0.446534


In [22]:
print(model_df['datetime'].min())
print(model_df['datetime'].max())

2006-12-23 17:00:00
2010-11-26 21:00:00


In [23]:
n = len(model_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = model_df.iloc[:train_end].copy()
val_df   = model_df.iloc[train_end:val_end].copy()
test_df  = model_df.iloc[val_end:].copy()

print("train:", train_df.shape, train_df['datetime'].min(), "to", train_df['datetime'].max())
print("val:  ", val_df.shape,   val_df['datetime'].min(),   "to", val_df['datetime'].max())
print("test: ", test_df.shape,  test_df['datetime'].min(),  "to", test_df['datetime'].max())

train: (24094, 35) 2006-12-23 17:00:00 to 2009-09-22 14:00:00
val:   (5163, 35) 2009-09-22 15:00:00 to 2010-04-25 17:00:00
test:  (5164, 35) 2010-04-25 18:00:00 to 2010-11-26 21:00:00


In [24]:
from sklearn.preprocessing import StandardScaler

# columns we will use as model inputs
feature_cols = [
    'target',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
    'hour_sin',
    'hour_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',
    'lag_1',
    'lag_24',
    'lag_48',
    'lag_168',
    'roll_mean_24',
    'roll_std_24',
    'roll_mean_168',
    'roll_std_168',
    'wd_0',
    'wd_1',
    'wd_2',
    'wd_3',
    'wd_4',
    'wd_5',
    'wd_6'
]

scaler = StandardScaler()

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

print(train_scaled[feature_cols].head())
print(train_scaled[feature_cols].shape)
print(val_scaled[feature_cols].shape)
print(test_scaled[feature_cols].shape)

     target  Global_reactive_power   Voltage  Global_intensity  \
0  3.066800               1.418940 -2.239090          4.847829   
1  2.362034              -0.322691 -0.810537          3.040604   
2  2.482355               1.259565 -0.571534          3.282543   
3  2.513486               0.052014 -0.640562          3.359172   
4  2.036446               1.715707 -0.615695          2.402608   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  hour_sin  hour_cos  \
0        4.175770       -0.157770        1.480061 -1.366132 -0.366092   
1       -0.318066       -0.306470        1.563089 -1.414321 -0.000071   
2       -0.318066       -0.173021        1.579233 -1.366132  0.365951   
3       -0.318066       -0.230214        1.572314 -1.224850  0.707029   
4       -0.318066       -0.260716       -0.053656 -1.000103  0.999919   

    day_sin  ...  roll_std_24  roll_mean_168  roll_std_168      wd_0  \
0 -1.412872  ...    -0.709545       1.760016      1.160860 -0.409216   
1 -1.412872  ...    

In [25]:
print(train_scaled[feature_cols].mean().head())
print(train_scaled[feature_cols].std().head())

target                   8.021401e-17
Global_reactive_power    7.549553e-17
Voltage                 -2.491353e-15
Global_intensity        -4.718471e-18
Sub_metering_1          -1.238599e-17
dtype: float64
target                   1.000021
Global_reactive_power    1.000021
Voltage                  1.000021
Global_intensity         1.000021
Sub_metering_1           1.000021
dtype: float64


In [26]:
INPUT_LEN = 192
HORIZON = 24

def make_windows(df, feature_cols, target_col='target', input_len=192, horizon=24):
    X, y = [], []
    feat_values = df[feature_cols].values
    target_values = df[target_col].values

    for i in range(len(df) - input_len - horizon + 1):
        x_i = feat_values[i:i + input_len]
        y_i = target_values[i + input_len:i + input_len + horizon]
        X.append(x_i)
        y.append(y_i)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train, y_train = make_windows(train_scaled, feature_cols, target_col='target', input_len=INPUT_LEN, horizon=HORIZON)
X_val, y_val     = make_windows(val_scaled, feature_cols, target_col='target', input_len=INPUT_LEN, horizon=HORIZON)
X_test, y_test   = make_windows(test_scaled, feature_cols, target_col='target', input_len=INPUT_LEN, horizon=HORIZON)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train: (23879, 192, 28) y_train: (23879, 24)
X_val:   (4948, 192, 28) y_val:   (4948, 24)
X_test:  (4949, 192, 28) y_test:  (4949, 24)


In [27]:

class LoadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = LoadDataset(X_train, y_train)
val_ds   = LoadDataset(X_val, y_val)
test_ds  = LoadDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))
print(next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape)

23879 4948 4949
torch.Size([64, 192, 28]) torch.Size([64, 24])


In [28]:


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", DEVICE)

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2, horizon=24):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, horizon)
        )

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]   # take last time step
        out = self.fc(last_hidden)
        return out

model = LSTMForecaster(
    input_size=X_train.shape[2],
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    horizon=24
).to(DEVICE)

print(model)

Using device: cpu
LSTMForecaster(
  (lstm): LSTM(28, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=24, bias=True)
  )
)


In [29]:

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10, device='cpu'):
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # ----- training -----
        model.train()
        running_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * X_batch.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        # ----- validation -----
        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)

                running_val_loss += loss.item() * X_batch.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, train_losses, val_losses

In [ ]:
model, train_losses, val_losses = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=10,
    device=DEVICE
)

Epoch 1/10 | Train Loss: 0.618888 | Val Loss: 0.509873
Epoch 2/10 | Train Loss: 0.462104 | Val Loss: 0.550073
Epoch 3/10 | Train Loss: 0.385549 | Val Loss: 0.569869
Epoch 4/10 | Train Loss: 0.330976 | Val Loss: 0.588102
Epoch 5/10 | Train Loss: 0.293211 | Val Loss: 0.579745
Epoch 6/10 | Train Loss: 0.268610 | Val Loss: 0.585317
Epoch 7/10 | Train Loss: 0.252206 | Val Loss: 0.585438
Epoch 8/10 | Train Loss: 0.238341 | Val Loss: 0.593901
Epoch 9/10 | Train Loss: 0.227301 | Val Loss: 0.616040


In [ ]:
model.eval()

preds = []
trues = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        y_pred = model(X_batch).cpu().numpy()
        preds.append(y_pred)
        trues.append(y_batch.numpy())

preds = np.concatenate(preds, axis=0)
trues = np.concatenate(trues, axis=0)

print("preds shape:", preds.shape)
print("trues shape:", trues.shape)

rmse_scaled = np.sqrt(mean_squared_error(trues.flatten(), preds.flatten()))
mae_scaled = mean_absolute_error(trues.flatten(), preds.flatten())

print("Scaled RMSE:", rmse_scaled)
print("Scaled MAE: ", mae_scaled)

In [ ]:
target_idx = feature_cols.index('target')

target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

preds_unscaled_log = preds * target_std + target_mean
trues_unscaled_log = trues * target_std + target_mean

preds_kw = np.expm1(preds_unscaled_log)
trues_kw = np.expm1(trues_unscaled_log)

rmse = np.sqrt(mean_squared_error(trues_kw.flatten(), preds_kw.flatten()))
mae = mean_absolute_error(trues_kw.flatten(), preds_kw.flatten())

mape = np.mean(
    np.abs((trues_kw.flatten() - preds_kw.flatten()) / np.clip(np.abs(trues_kw.flatten()), 1e-6, None))
) * 100

print("Original-scale RMSE:", rmse)
print("Original-scale MAE: ", mae)
print("Original-scale MAPE:", mape)

In [ ]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)

def train_model_early_stop(model, train_loader, val_loader, criterion, optimizer,
                           epochs=20, patience=3, device='cpu'):
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses = []
    wait = 0

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_train_loss += loss.item() * X_batch.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                running_val_loss += loss.item() * X_batch.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_model_wts)
    return model, train_losses, val_losses

In [ ]:
model = LSTMForecaster(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    dropout=0.3,
    horizon=24
).to(DEVICE)

criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)

model, train_losses, val_losses = train_model_early_stop(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=20,
    patience=3,
    device=DEVICE
)

In [ ]:
model.eval()

preds = []
trues = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        y_pred = model(X_batch).cpu().numpy()
        preds.append(y_pred)
        trues.append(y_batch.numpy())

preds = np.concatenate(preds, axis=0)
trues = np.concatenate(trues, axis=0)

print("preds shape:", preds.shape)
print("trues shape:", trues.shape)

rmse_scaled = np.sqrt(mean_squared_error(trues.flatten(), preds.flatten()))
mae_scaled = mean_absolute_error(trues.flatten(), preds.flatten())

print("Scaled RMSE:", rmse_scaled)
print("Scaled MAE: ", mae_scaled)

In [ ]:
target_idx = feature_cols.index('target')

target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

preds_unscaled_log = preds * target_std + target_mean
trues_unscaled_log = trues * target_std + target_mean

preds_kw = np.expm1(preds_unscaled_log)
trues_kw = np.expm1(trues_unscaled_log)

rmse = np.sqrt(mean_squared_error(trues_kw.flatten(), preds_kw.flatten()))
mae = mean_absolute_error(trues_kw.flatten(), preds_kw.flatten())

mape = np.mean(
    np.abs((trues_kw.flatten() - preds_kw.flatten()) / np.clip(np.abs(trues_kw.flatten()), 1e-6, None))
) * 100

print("Original-scale RMSE:", rmse)
print("Original-scale MAE: ", mae)
print("Original-scale MAPE:", mape)

In [ ]:
def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return 100 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

smape_value = smape(trues_kw.flatten(), preds_kw.flatten())
print("sMAPE:", smape_value)

In [ ]:
true_flat = trues_kw.flatten()

print("Min true load:", true_flat.min())
print("Fraction < 0.1 kW:", np.mean(true_flat < 0.1))
print("Fraction < 0.2 kW:", np.mean(true_flat < 0.2))
print("Fraction < 0.5 kW:", np.mean(true_flat < 0.5))

In [ ]:
model_df2 = model_df.copy()

month = model_df2['datetime'].dt.month

model_df2['season'] = np.where(
    month.isin([12, 1, 2]), 'winter',
    np.where(month.isin([6, 7, 8]), 'summer', 'other')
)

print(model_df2['season'].value_counts())
print(model_df2[['datetime', 'season']].head(10))

In [ ]:
summer_df = model_df2[model_df2['season'] == 'summer'].copy().reset_index(drop=True)

print(summer_df.shape)
print(summer_df['datetime'].min())
print(summer_df['datetime'].max())
summer_df.head()

In [ ]:
n_summer = len(summer_df)

summer_train_end = int(n_summer * 0.70)
summer_val_end = int(n_summer * 0.85)

summer_train_df = summer_df.iloc[:summer_train_end].copy()
summer_val_df   = summer_df.iloc[summer_train_end:summer_val_end].copy()
summer_test_df  = summer_df.iloc[summer_val_end:].copy()

print("summer train:", summer_train_df.shape, summer_train_df['datetime'].min(), "to", summer_train_df['datetime'].max())
print("summer val:  ", summer_val_df.shape,   summer_val_df['datetime'].min(),   "to", summer_val_df['datetime'].max())
print("summer test: ", summer_test_df.shape,  summer_test_df['datetime'].min(),  "to", summer_test_df['datetime'].max())

In [ ]:
summer_feature_cols = feature_cols.copy()

summer_scaler = StandardScaler()

summer_train_scaled = summer_train_df.copy()
summer_val_scaled   = summer_val_df.copy()
summer_test_scaled  = summer_test_df.copy()

summer_train_scaled[summer_feature_cols] = summer_scaler.fit_transform(summer_train_df[summer_feature_cols])
summer_val_scaled[summer_feature_cols]   = summer_scaler.transform(summer_val_df[summer_feature_cols])
summer_test_scaled[summer_feature_cols]  = summer_scaler.transform(summer_test_df[summer_feature_cols])

print(summer_train_scaled[summer_feature_cols].shape)
print(summer_val_scaled[summer_feature_cols].shape)
print(summer_test_scaled[summer_feature_cols].shape)

print(summer_train_scaled[summer_feature_cols].mean().head())
print(summer_train_scaled[summer_feature_cols].std().head())

In [ ]:
X_train_summer, y_train_summer = make_windows(
    summer_train_scaled,
    summer_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

X_val_summer, y_val_summer = make_windows(
    summer_val_scaled,
    summer_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

X_test_summer, y_test_summer = make_windows(
    summer_test_scaled,
    summer_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

print("X_train_summer:", X_train_summer.shape, "y_train_summer:", y_train_summer.shape)
print("X_val_summer:  ", X_val_summer.shape, "y_val_summer:  ", y_val_summer.shape)
print("X_test_summer: ", X_test_summer.shape, "y_test_summer: ", y_test_summer.shape)

In [ ]:
summer_train_ds = LoadDataset(X_train_summer, y_train_summer)
summer_val_ds   = LoadDataset(X_val_summer, y_val_summer)
summer_test_ds  = LoadDataset(X_test_summer, y_test_summer)

summer_train_loader = DataLoader(summer_train_ds, batch_size=64, shuffle=True)
summer_val_loader   = DataLoader(summer_val_ds, batch_size=64, shuffle=False)
summer_test_loader  = DataLoader(summer_test_ds, batch_size=64, shuffle=False)

print(len(summer_train_ds), len(summer_val_ds), len(summer_test_ds))
print(next(iter(summer_train_loader))[0].shape, next(iter(summer_train_loader))[1].shape)

In [ ]:
summer_model = LSTMForecaster(
    input_size=X_train_summer.shape[2],
    hidden_size=64,
    num_layers=2,
    dropout=0.3,
    horizon=24
).to(DEVICE)

summer_criterion = nn.HuberLoss(delta=1.0)
summer_optimizer = torch.optim.Adam(
    summer_model.parameters(),
    lr=5e-4,
    weight_decay=1e-5
)

summer_model, summer_train_losses, summer_val_losses = train_model_early_stop(
    summer_model,
    summer_train_loader,
    summer_val_loader,
    summer_criterion,
    summer_optimizer,
    epochs=20,
    patience=3,
    device=DEVICE
)

In [ ]:
summer_model.eval()

summer_preds = []
summer_trues = []

with torch.no_grad():
    for X_batch, y_batch in summer_test_loader:
        X_batch = X_batch.to(DEVICE)
        y_pred = summer_model(X_batch).cpu().numpy()
        summer_preds.append(y_pred)
        summer_trues.append(y_batch.numpy())

summer_preds = np.concatenate(summer_preds, axis=0)
summer_trues = np.concatenate(summer_trues, axis=0)

print("summer_preds shape:", summer_preds.shape)
print("summer_trues shape:", summer_trues.shape)

summer_rmse_scaled = np.sqrt(mean_squared_error(summer_trues.flatten(), summer_preds.flatten()))
summer_mae_scaled = mean_absolute_error(summer_trues.flatten(), summer_preds.flatten())

print("Summer scaled RMSE:", summer_rmse_scaled)
print("Summer scaled MAE: ", summer_mae_scaled)

In [ ]:
summer_target_idx = summer_feature_cols.index('target')

summer_target_mean = summer_scaler.mean_[summer_target_idx]
summer_target_std = summer_scaler.scale_[summer_target_idx]

summer_preds_unscaled_log = summer_preds * summer_target_std + summer_target_mean
summer_trues_unscaled_log = summer_trues * summer_target_std + summer_target_mean

summer_preds_kw = np.expm1(summer_preds_unscaled_log)
summer_trues_kw = np.expm1(summer_trues_unscaled_log)

summer_rmse = np.sqrt(mean_squared_error(summer_trues_kw.flatten(), summer_preds_kw.flatten()))
summer_mae = mean_absolute_error(summer_trues_kw.flatten(), summer_preds_kw.flatten())

summer_mape = np.mean(
    np.abs((summer_trues_kw.flatten() - summer_preds_kw.flatten()) / np.clip(np.abs(summer_trues_kw.flatten()), 1e-6, None))
) * 100

print("Summer original-scale RMSE:", summer_rmse)
print("Summer original-scale MAE: ", summer_mae)
print("Summer original-scale MAPE:", summer_mape)

In [ ]:
winter_df = model_df2[model_df2['season'] == 'winter'].copy().reset_index(drop=True)

print(winter_df.shape)
print(winter_df['datetime'].min())
print(winter_df['datetime'].max())
winter_df.head()

In [ ]:
n_winter = len(winter_df)

winter_train_end = int(n_winter * 0.70)
winter_val_end = int(n_winter * 0.85)

winter_train_df = winter_df.iloc[:winter_train_end].copy()
winter_val_df   = winter_df.iloc[winter_train_end:winter_val_end].copy()
winter_test_df  = winter_df.iloc[winter_val_end:].copy()

print("winter train:", winter_train_df.shape, winter_train_df['datetime'].min(), "to", winter_train_df['datetime'].max())
print("winter val:  ", winter_val_df.shape,   winter_val_df['datetime'].min(),   "to", winter_val_df['datetime'].max())
print("winter test: ", winter_test_df.shape,  winter_test_df['datetime'].min(),  "to", winter_test_df['datetime'].max())

In [ ]:
winter_feature_cols = feature_cols.copy()

winter_scaler = StandardScaler()

winter_train_scaled = winter_train_df.copy()
winter_val_scaled   = winter_val_df.copy()
winter_test_scaled  = winter_test_df.copy()

winter_train_scaled[winter_feature_cols] = winter_scaler.fit_transform(winter_train_df[winter_feature_cols])
winter_val_scaled[winter_feature_cols]   = winter_scaler.transform(winter_val_df[winter_feature_cols])
winter_test_scaled[winter_feature_cols]  = winter_scaler.transform(winter_test_df[winter_feature_cols])

print(winter_train_scaled[winter_feature_cols].shape)
print(winter_val_scaled[winter_feature_cols].shape)
print(winter_test_scaled[winter_feature_cols].shape)

print(winter_train_scaled[winter_feature_cols].mean().head())
print(winter_train_scaled[winter_feature_cols].std().head())

In [ ]:
X_train_winter, y_train_winter = make_windows(
    winter_train_scaled,
    winter_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

X_val_winter, y_val_winter = make_windows(
    winter_val_scaled,
    winter_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

X_test_winter, y_test_winter = make_windows(
    winter_test_scaled,
    winter_feature_cols,
    target_col='target',
    input_len=INPUT_LEN,
    horizon=HORIZON
)

print("X_train_winter:", X_train_winter.shape, "y_train_winter:", y_train_winter.shape)
print("X_val_winter:  ", X_val_winter.shape, "y_val_winter:  ", y_val_winter.shape)
print("X_test_winter: ", X_test_winter.shape, "y_test_winter: ", y_test_winter.shape)

In [ ]:
winter_train_ds = LoadDataset(X_train_winter, y_train_winter)
winter_val_ds   = LoadDataset(X_val_winter, y_val_winter)
winter_test_ds  = LoadDataset(X_test_winter, y_test_winter)

winter_train_loader = DataLoader(winter_train_ds, batch_size=64, shuffle=True)
winter_val_loader   = DataLoader(winter_val_ds, batch_size=64, shuffle=False)
winter_test_loader  = DataLoader(winter_test_ds, batch_size=64, shuffle=False)

print(len(winter_train_ds), len(winter_val_ds), len(winter_test_ds))
print(next(iter(winter_train_loader))[0].shape, next(iter(winter_train_loader))[1].shape)

In [ ]:
winter_model = LSTMForecaster(
    input_size=X_train_winter.shape[2],
    hidden_size=64,
    num_layers=2,
    dropout=0.3,
    horizon=24
).to(DEVICE)

winter_criterion = nn.HuberLoss(delta=1.0)
winter_optimizer = torch.optim.Adam(
    winter_model.parameters(),
    lr=5e-4,
    weight_decay=1e-5
)

winter_model, winter_train_losses, winter_val_losses = train_model_early_stop(
    winter_model,
    winter_train_loader,
    winter_val_loader,
    winter_criterion,
    winter_optimizer,
    epochs=20,
    patience=3,
    device=DEVICE
)

In [ ]:
winter_model.eval()

winter_preds = []
winter_trues = []

with torch.no_grad():
    for X_batch, y_batch in winter_test_loader:
        X_batch = X_batch.to(DEVICE)
        y_pred = winter_model(X_batch).cpu().numpy()
        winter_preds.append(y_pred)
        winter_trues.append(y_batch.numpy())

winter_preds = np.concatenate(winter_preds, axis=0)
winter_trues = np.concatenate(winter_trues, axis=0)

print("winter_preds shape:", winter_preds.shape)
print("winter_trues shape:", winter_trues.shape)

winter_rmse_scaled = np.sqrt(mean_squared_error(winter_trues.flatten(), winter_preds.flatten()))
winter_mae_scaled = mean_absolute_error(winter_trues.flatten(), winter_preds.flatten())

print("Winter scaled RMSE:", winter_rmse_scaled)
print("Winter scaled MAE: ", winter_mae_scaled)

In [ ]:
winter_target_idx = winter_feature_cols.index('target')

winter_target_mean = winter_scaler.mean_[winter_target_idx]
winter_target_std = winter_scaler.scale_[winter_target_idx]

winter_preds_unscaled_log = winter_preds * winter_target_std + winter_target_mean
winter_trues_unscaled_log = winter_trues * winter_target_std + winter_target_mean

winter_preds_kw = np.expm1(winter_preds_unscaled_log)
winter_trues_kw = np.expm1(winter_trues_unscaled_log)

winter_rmse = np.sqrt(mean_squared_error(winter_trues_kw.flatten(), winter_preds_kw.flatten()))
winter_mae = mean_absolute_error(winter_trues_kw.flatten(), winter_preds_kw.flatten())

winter_mape = np.mean(
    np.abs((winter_trues_kw.flatten() - winter_preds_kw.flatten()) / np.clip(np.abs(winter_trues_kw.flatten()), 1e-6, None))
) * 100

print("Winter original-scale RMSE:", winter_rmse)
print("Winter original-scale MAE: ", winter_mae)
print("Winter original-scale MAPE:", winter_mape)

In [ ]:
results_df = pd.DataFrame({
    'model': ['all_season', 'summer_only', 'winter_only'],
    'RMSE': [0.5938753250109605, 0.500829232063017, 0.7926112882648556],
    'MAE':  [0.4206670947001056, 0.3356997745493183, 0.5819381981558279],
    'MAPE': [54.17787234086738, 60.47003695097042, 56.73434859161859]
})

print(results_df)

In [ ]:
import matplotlib.pyplot as plt

results_df.plot(x='model', y=['RMSE', 'MAE', 'MAPE'], kind='bar', figsize=(10, 5))
plt.title('Model comparison')
plt.xticks(rotation=0)
plt.show()

In [ ]:
feat3 = feat.copy()

feat3['unmetered_load'] = (
    feat3['Global_active_power'] * 1000 / 60
    - feat3['Sub_metering_1']
    - feat3['Sub_metering_2']
    - feat3['Sub_metering_3']
)

# avoid negative values caused by rounding/noise
feat3['unmetered_load'] = feat3['unmetered_load'].clip(lower=0)

print(feat3[['Global_active_power', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'unmetered_load']].head())
print(feat3['unmetered_load'].describe())

In [ ]:
print(feat3.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
feat4 = feat3.copy()

# log target
feat4['target'] = np.log1p(feat4['Global_active_power'])

# lag features from target
feat4['lag_1'] = feat4['target'].shift(1)
feat4['lag_24'] = feat4['target'].shift(24)
feat4['lag_48'] = feat4['target'].shift(48)
feat4['lag_168'] = feat4['target'].shift(168)

# rolling features from target
feat4['roll_mean_24'] = feat4['target'].rolling(24).mean()
feat4['roll_std_24'] = feat4['target'].rolling(24).std()
feat4['roll_mean_168'] = feat4['target'].rolling(168).mean()
feat4['roll_std_168'] = feat4['target'].rolling(168).std()

# drop rows made incomplete by lags/rolling
model_df4 = feat4.dropna().reset_index(drop=True)

# season labels again
month = model_df4['datetime'].dt.month
model_df4['season'] = np.where(
    month.isin([12, 1, 2]), 'winter',
    np.where(month.isin([6, 7, 8]), 'summer', 'other')
)

print(model_df4.shape)
print(model_df4[['datetime', 'target', 'unmetered_load', 'season']].head())
print(model_df4.isna().sum().sum())

In [ ]:
feature_cols_v2 = [
    'target',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
    'unmetered_load',
    'hour_sin',
    'hour_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',
    'lag_1',
    'lag_24',
    'lag_48',
    'lag_168',
    'roll_mean_24',
    'roll_std_24',
    'roll_mean_168',
    'roll_std_168',
    'wd_0',
    'wd_1',
    'wd_2',
    'wd_3',
    'wd_4',
    'wd_5',
    'wd_6'
]

print("Number of features:", len(feature_cols_v2))
print(feature_cols_v2)

In [ ]:
winter_df_v2 = model_df4[model_df4['season'] == 'winter'].copy().reset_index(drop=True)

print(winter_df_v2.shape)
print(winter_df_v2['datetime'].min())
print(winter_df_v2['datetime'].max())
print(winter_df_v2[['datetime', 'unmetered_load', 'target']].head())
print("Total missing values:", winter_df_v2.isna().sum().sum())

In [ ]:
# 1) time split
n_winter_v2 = len(winter_df_v2)

winter_v2_train_end = int(n_winter_v2 * 0.70)
winter_v2_val_end = int(n_winter_v2 * 0.85)

winter_v2_train_df = winter_df_v2.iloc[:winter_v2_train_end].copy()
winter_v2_val_df   = winter_df_v2.iloc[winter_v2_train_end:winter_v2_val_end].copy()
winter_v2_test_df  = winter_df_v2.iloc[winter_v2_val_end:].copy()

print("winter_v2 train:", winter_v2_train_df.shape, winter_v2_train_df['datetime'].min(), "to", winter_v2_train_df['datetime'].max())
print("winter_v2 val:  ", winter_v2_val_df.shape,   winter_v2_val_df['datetime'].min(),   "to", winter_v2_val_df['datetime'].max())
print("winter_v2 test: ", winter_v2_test_df.shape,  winter_v2_test_df['datetime'].min(),  "to", winter_v2_test_df['datetime'].max())

# 2) scale
from sklearn.preprocessing import StandardScaler

winter_scaler_v2 = StandardScaler()

winter_v2_train_scaled = winter_v2_train_df.copy()
winter_v2_val_scaled   = winter_v2_val_df.copy()
winter_v2_test_scaled  = winter_v2_test_df.copy()

winter_v2_train_scaled[feature_cols_v2] = winter_scaler_v2.fit_transform(winter_v2_train_df[feature_cols_v2])
winter_v2_val_scaled[feature_cols_v2]   = winter_scaler_v2.transform(winter_v2_val_df[feature_cols_v2])
winter_v2_test_scaled[feature_cols_v2]  = winter_scaler_v2.transform(winter_v2_test_df[feature_cols_v2])

print(winter_v2_train_scaled[feature_cols_v2].shape)
print(winter_v2_val_scaled[feature_cols_v2].shape)
print(winter_v2_test_scaled[feature_cols_v2].shape)

# 3) build windows
X_train_winter_v2, y_train_winter_v2 = make_windows(
    winter_v2_train_scaled, feature_cols_v2, target_col='target',
    input_len=INPUT_LEN, horizon=HORIZON
)
X_val_winter_v2, y_val_winter_v2 = make_windows(
    winter_v2_val_scaled, feature_cols_v2, target_col='target',
    input_len=INPUT_LEN, horizon=HORIZON
)
X_test_winter_v2, y_test_winter_v2 = make_windows(
    winter_v2_test_scaled, feature_cols_v2, target_col='target',
    input_len=INPUT_LEN, horizon=HORIZON
)

print("X_train_winter_v2:", X_train_winter_v2.shape, "y_train_winter_v2:", y_train_winter_v2.shape)
print("X_val_winter_v2:  ", X_val_winter_v2.shape, "y_val_winter_v2:  ", y_val_winter_v2.shape)
print("X_test_winter_v2: ", X_test_winter_v2.shape, "y_test_winter_v2: ", y_test_winter_v2.shape)

In [ ]:
winter_train_ds_v2 = LoadDataset(X_train_winter_v2, y_train_winter_v2)
winter_val_ds_v2   = LoadDataset(X_val_winter_v2, y_val_winter_v2)
winter_test_ds_v2  = LoadDataset(X_test_winter_v2, y_test_winter_v2)

winter_train_loader_v2 = DataLoader(winter_train_ds_v2, batch_size=64, shuffle=True)
winter_val_loader_v2   = DataLoader(winter_val_ds_v2, batch_size=64, shuffle=False)
winter_test_loader_v2  = DataLoader(winter_test_ds_v2, batch_size=64, shuffle=False)

winter_model_v2 = LSTMForecaster(
    input_size=X_train_winter_v2.shape[2],
    hidden_size=64,
    num_layers=2,
    dropout=0.3,
    horizon=24
).to(DEVICE)

winter_criterion_v2 = nn.HuberLoss(delta=1.0)
winter_optimizer_v2 = torch.optim.Adam(
    winter_model_v2.parameters(),
    lr=5e-4,
    weight_decay=1e-5
)

winter_model_v2, winter_train_losses_v2, winter_val_losses_v2 = train_model_early_stop(
    winter_model_v2,
    winter_train_loader_v2,
    winter_val_loader_v2,
    winter_criterion_v2,
    winter_optimizer_v2,
    epochs=20,
    patience=3,
    device=DEVICE
)

In [ ]:

winter_model_v2.eval()

winter_preds_v2 = []
winter_trues_v2 = []

with torch.no_grad():
    for X_batch, y_batch in winter_test_loader_v2:
        X_batch = X_batch.to(DEVICE)
        y_pred = winter_model_v2(X_batch).cpu().numpy()
        winter_preds_v2.append(y_pred)
        winter_trues_v2.append(y_batch.numpy())

winter_preds_v2 = np.concatenate(winter_preds_v2, axis=0)
winter_trues_v2 = np.concatenate(winter_trues_v2, axis=0)

print("winter_preds_v2 shape:", winter_preds_v2.shape)
print("winter_trues_v2 shape:", winter_trues_v2.shape)

winter_rmse_scaled_v2 = np.sqrt(mean_squared_error(winter_trues_v2.flatten(), winter_preds_v2.flatten()))
winter_mae_scaled_v2 = mean_absolute_error(winter_trues_v2.flatten(), winter_preds_v2.flatten())

print("Winter v2 scaled RMSE:", winter_rmse_scaled_v2)
print("Winter v2 scaled MAE: ", winter_mae_scaled_v2)

In [ ]:
winter_target_idx_v2 = feature_cols_v2.index('target')

winter_target_mean_v2 = winter_scaler_v2.mean_[winter_target_idx_v2]
winter_target_std_v2 = winter_scaler_v2.scale_[winter_target_idx_v2]

winter_preds_unscaled_log_v2 = winter_preds_v2 * winter_target_std_v2 + winter_target_mean_v2
winter_trues_unscaled_log_v2 = winter_trues_v2 * winter_target_std_v2 + winter_target_mean_v2

winter_preds_kw_v2 = np.expm1(winter_preds_unscaled_log_v2)
winter_trues_kw_v2 = np.expm1(winter_trues_unscaled_log_v2)

winter_rmse_v2 = np.sqrt(mean_squared_error(winter_trues_kw_v2.flatten(), winter_preds_kw_v2.flatten()))
winter_mae_v2 = mean_absolute_error(winter_trues_kw_v2.flatten(), winter_preds_kw_v2.flatten())

winter_mape_v2 = np.mean(
    np.abs((winter_trues_kw_v2.flatten() - winter_preds_kw_v2.flatten()) / np.clip(np.abs(winter_trues_kw_v2.flatten()), 1e-6, None))
) * 100

print("Winter v2 original-scale RMSE:", winter_rmse_v2)
print("Winter v2 original-scale MAE: ", winter_mae_v2)
print("Winter v2 original-scale MAPE:", winter_mape_v2)

In [ ]:
winter_model_v3 = LSTMForecaster(
    input_size=X_train_winter.shape[2],   # use the original winter set first
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    horizon=24
).to(DEVICE)

winter_criterion_v3 = nn.HuberLoss(delta=1.0)
winter_optimizer_v3 = torch.optim.Adam(
    winter_model_v3.parameters(),
    lr=3e-4,
    weight_decay=1e-5
)

winter_model_v3, winter_train_losses_v3, winter_val_losses_v3 = train_model_early_stop(
    winter_model_v3,
    winter_train_loader,
    winter_val_loader,
    winter_criterion_v3,
    winter_optimizer_v3,
    epochs=20,
    patience=4,
    device=DEVICE
)

In [ ]:

winter_model_v3.eval()

winter_preds_v3 = []
winter_trues_v3 = []

with torch.no_grad():
    for X_batch, y_batch in winter_test_loader:
        X_batch = X_batch.to(DEVICE)
        y_pred = winter_model_v3(X_batch).cpu().numpy()
        winter_preds_v3.append(y_pred)
        winter_trues_v3.append(y_batch.numpy())

winter_preds_v3 = np.concatenate(winter_preds_v3, axis=0)
winter_trues_v3 = np.concatenate(winter_trues_v3, axis=0)

print("winter_preds_v3 shape:", winter_preds_v3.shape)
print("winter_trues_v3 shape:", winter_trues_v3.shape)

winter_rmse_scaled_v3 = np.sqrt(mean_squared_error(winter_trues_v3.flatten(), winter_preds_v3.flatten()))
winter_mae_scaled_v3 = mean_absolute_error(winter_trues_v3.flatten(), winter_preds_v3.flatten())

print("Winter v3 scaled RMSE:", winter_rmse_scaled_v3)
print("Winter v3 scaled MAE: ", winter_mae_scaled_v3)

In [ ]:
X_train_winter_v2_flat = X_train_winter_v2.reshape(X_train_winter_v2.shape[0], -1)
X_val_winter_v2_flat   = X_val_winter_v2.reshape(X_val_winter_v2.shape[0], -1)
X_test_winter_v2_flat  = X_test_winter_v2.reshape(X_test_winter_v2.shape[0], -1)

print(X_train_winter_v2_flat.shape, X_test_winter_v2_flat.shape)

In [ ]:
winter_target_feature_idx_v2 = feature_cols_v2.index('target')

def naive_previous_24h(X, target_feature_idx):
    return X[:, -24:, target_feature_idx]

y_pred_naive_winter_v2 = naive_previous_24h(X_test_winter_v2, winter_target_feature_idx_v2)
print(y_pred_naive_winter_v2.shape)

In [ ]:
lr_winter_v2 = LinearRegression()
lr_winter_v2.fit(X_train_winter_v2_flat, y_train_winter_v2)
y_pred_lr_winter_v2 = lr_winter_v2.predict(X_test_winter_v2_flat)

print(y_pred_lr_winter_v2.shape)

In [ ]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

xgb_winter_v2 = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
)

xgb_winter_v2.fit(X_train_winter_v2_flat, y_train_winter_v2)
y_pred_xgb_winter_v2 = xgb_winter_v2.predict(X_test_winter_v2_flat)

print(y_pred_xgb_winter_v2.shape)

In [ ]:
winter_target_idx_v2 = feature_cols_v2.index('target')
winter_target_mean_v2 = winter_scaler_v2.mean_[winter_target_idx_v2]
winter_target_std_v2  = winter_scaler_v2.scale_[winter_target_idx_v2]

def invert_target_from_global_scaler(y_scaled, mean, std):
    y_unscaled_log = y_scaled * std + mean
    y_original = np.expm1(y_unscaled_log)
    return y_original

In [ ]:
winter_preds_kw_v3 = invert_target_from_global_scaler(
    winter_preds_v3, winter_target_mean_v2, winter_target_std_v2
)

winter_trues_kw_v3 = invert_target_from_global_scaler(
    winter_trues_v3, winter_target_mean_v2, winter_target_std_v2
)

In [ ]:
winter_preds_kw_naive_v2 = invert_target_from_global_scaler(
    y_pred_naive_winter_v2, winter_target_mean_v2, winter_target_std_v2
)

winter_preds_kw_lr_v2 = invert_target_from_global_scaler(
    y_pred_lr_winter_v2, winter_target_mean_v2, winter_target_std_v2
)

winter_preds_kw_xgb_v2 = invert_target_from_global_scaler(
    y_pred_xgb_winter_v2, winter_target_mean_v2, winter_target_std_v2
)

In [ ]:
def mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

def smape(y_true, y_pred, eps=1e-6):
    denom = np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100

def regression_metrics(y_true, y_pred):
    y_true_f = y_true.reshape(-1)
    y_pred_f = y_pred.reshape(-1)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true_f, y_pred_f)),
        "MAE": mean_absolute_error(y_true_f, y_pred_f),
        "MAPE": mape(y_true_f, y_pred_f),
        "sMAPE": smape(y_true_f, y_pred_f),
    }

results_winter_compare = []

results_winter_compare.append({
    "Model": "Naive 24h Repeat",
    **regression_metrics(winter_trues_kw_v3, winter_preds_kw_naive_v2)
})

results_winter_compare.append({
    "Model": "Linear Regression",
    **regression_metrics(winter_trues_kw_v3, winter_preds_kw_lr_v2)
})

results_winter_compare.append({
    "Model": "XGBoost",
    **regression_metrics(winter_trues_kw_v3, winter_preds_kw_xgb_v2)
})

results_winter_compare.append({
    "Model": "LSTM Winter v3",
    **regression_metrics(winter_trues_kw_v3, winter_preds_kw_v3)
})

results_winter_compare_df = pd.DataFrame(results_winter_compare).sort_values("RMSE").reset_index(drop=True)
results_winter_compare_df

In [ ]:
winter_target_idx = winter_feature_cols.index('target')

winter_target_mean = winter_scaler.mean_[winter_target_idx]
winter_target_std = winter_scaler.scale_[winter_target_idx]

winter_preds_unscaled_log_v3 = winter_preds_v3 * winter_target_std + winter_target_mean
winter_trues_unscaled_log_v3 = winter_trues_v3 * winter_target_std + winter_target_mean

winter_preds_kw_v3 = np.expm1(winter_preds_unscaled_log_v3)
winter_trues_kw_v3 = np.expm1(winter_trues_unscaled_log_v3)

winter_rmse_v3 = np.sqrt(mean_squared_error(winter_trues_kw_v3.flatten(), winter_preds_kw_v3.flatten()))
winter_mae_v3 = mean_absolute_error(winter_trues_kw_v3.flatten(), winter_preds_kw_v3.flatten())

winter_mape_v3 = np.mean(
    np.abs((winter_trues_kw_v3.flatten() - winter_preds_kw_v3.flatten()) / np.clip(np.abs(winter_trues_kw_v3.flatten()), 1e-6, None))
) * 100

print("Winter v3 original-scale RMSE:", winter_rmse_v3)
print("Winter v3 original-scale MAE: ", winter_mae_v3)
print("Winter v3 original-scale MAPE:", winter_mape_v3)

In [ ]:
final_results = pd.DataFrame({
    'model': [
        'all_season_improved',
        'summer_only',
        'winter_only',
        'winter_v2_unmetered',
        'winter_v3_bigger_lstm'
    ],
    'RMSE': [
        0.5938753250109605,
        0.500829232063017,
        0.7926112882648556,
        0.7925391478834884,
        0.7968111724333168
    ],
    'MAE': [
        0.4206670947001056,
        0.3356997745493183,
        0.5819381981558279,
        0.5856552328747602,
        0.5858534523202863
    ],
    'MAPE': [
        54.17787234086738,
        60.47003695097042,
        56.73434859161859,
        57.37768729063545,
        55.66354716234342
    ]
})

print(final_results.sort_values('RMSE'))

In [ ]:
final_results.set_index('model')[['RMSE', 'MAE', 'MAPE']].plot(
    kind='bar',
    figsize=(12, 5)
)
plt.title('Final model comparison')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Results and comparison with the paper

In this project, I used the Individual Household Electric Power Consumption dataset and built several LSTM-based forecasting models to predict the next 24 hours from the previous 192 hours of hourly data. The preprocessing pipeline included missing-value interpolation, hourly resampling, logarithmic transformation of the target, calendar-based cyclical features, lag features, and rolling statistics.

### Models tested
1. All-season LSTM baseline with improved training
2. Summer-only LSTM
3. Winter-only LSTM
4. Winter-only LSTM with unmetered load feature
5. Winter-only larger LSTM

### Final results
- All-season improved: RMSE = 0.5939, MAE = 0.4207, MAPE = 54.18%
- Summer-only: RMSE = 0.5008, MAE = 0.3357, MAPE = 60.47%
- Winter-only: RMSE = 0.7926, MAE = 0.5819, MAPE = 56.73%
- Winter v2 with unmetered feature: RMSE = 0.7925, MAE = 0.5857, MAPE = 57.38%
- Winter v3 larger LSTM: RMSE = 0.7968, MAE = 0.5859, MAPE = 55.66%

### Interpretation
The summer-only model achieved the best RMSE and MAE, showing that seasonal specialization improved forecast accuracy for this dataset. Winter was consistently harder to predict than summer. This is consistent with the paper’s finding that forecasting performance differs by season and that winter can be more challenging.

However, my results are not directly comparable to the paper’s best-performing method because the paper uses a large multi-household dataset, then applies dimensionality reduction and clustering before training separate neural models. In contrast, the dataset used here contains only one household, so clustering-based improvement is not applicable in the same way.

### Comparison to the paper
The paper reports that LSTM was the best neural model among those tested, and that the strongest overall setup combined kernel PCA, k-means clustering with 4 clusters, and LSTM forecasting. The paper also used 192 hours of input to forecast the next 24 hours, which I followed here. My work reproduced the seasonal modeling idea and confirmed that season-specific models can help, especially in summer. But I could not reproduce the paper’s full clustering pipeline because the dataset structure is different.

### Conclusion
For this one-household dataset, seasonal LSTM modeling improved forecasting performance, especially for summer. The best overall model by RMSE and MAE was the summer-only LSTM. The next logical improvement would be to test this pipeline on a true multi-household dataset so that dimensionality reduction and clustering can be applied as in the paper.

In [ ]:
summer_true_flat = summer_trues_kw.flatten()
summer_pred_flat = summer_preds_kw.flatten()

plt.figure(figsize=(14, 5))
plt.plot(summer_true_flat[:500], label='Actual')
plt.plot(summer_pred_flat[:500], label='Predicted')
plt.title('Summer-only model: Actual vs Predicted load (first 500 forecast points)')
plt.xlabel('Forecasted hourly points')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(summer_true_flat[:2000], summer_pred_flat[:2000], alpha=0.4)
plt.xlabel('Actual load (kW)')
plt.ylabel('Predicted load (kW)')
plt.title('Summer-only model: Predicted vs Actual scatter')
plt.show()

In [ ]:

summer_residuals = summer_pred_flat - summer_true_flat

plt.figure(figsize=(14, 5))
plt.plot(summer_residuals[:500])
plt.axhline(0, linestyle='--')
plt.title('Summer-only model: Residuals (Predicted - Actual) for first 500 forecast points')
plt.xlabel('Forecasted hourly points')
plt.ylabel('Residual (kW)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(summer_residuals, bins=50)
plt.title('Summer-only model: Distribution of residuals')
plt.xlabel('Residual (kW)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sample_idx = 100   # you can change this later

plt.figure(figsize=(10, 5))
plt.plot(range(1, 25), summer_trues_kw[sample_idx], marker='o', label='Actual')
plt.plot(range(1, 25), summer_preds_kw[sample_idx], marker='o', label='Predicted')
plt.title(f'Summer-only model: 24-hour forecast example (sample {sample_idx})')
plt.xlabel('Hour ahead')
plt.ylabel('Global Active Power (kW)')
plt.xticks(range(1, 25, 2))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
for sample_idx in [0, 100, 300]:
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, 25), summer_trues_kw[sample_idx], marker='o', label='Actual')
    plt.plot(range(1, 25), summer_preds_kw[sample_idx], marker='o', label='Predicted')
    plt.title(f'Summer-only model: 24-hour forecast example (sample {sample_idx})')
    plt.xlabel('Hour ahead')
    plt.ylabel('Global Active Power (kW)')
    plt.xticks(range(1, 25, 2))
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
season_compare = pd.DataFrame({
    'season': ['summer', 'winter'],
    'RMSE': [summer_rmse, winter_rmse],
    'MAE': [summer_mae, winter_mae],
    'MAPE': [summer_mape, winter_mape]
})

print(season_compare)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['RMSE'])
plt.title('RMSE comparison: Summer vs Winter')
plt.ylabel('RMSE')
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['MAE'])
plt.title('MAE comparison: Summer vs Winter')
plt.ylabel('MAE')
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['MAPE'])
plt.title('MAPE comparison: Summer vs Winter')
plt.ylabel('MAPE (%)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(summer_train_losses, marker='o', label='Train Loss')
plt.plot(summer_val_losses, marker='o', label='Validation Loss')
plt.title('Summer-only model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(winter_train_losses, marker='o', label='Train Loss')
plt.plot(winter_val_losses, marker='o', label='Validation Loss')
plt.title('Winter-only model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
heatmap_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
    'target',
    'lag_1',
    'lag_24',
    'lag_168',
    'roll_mean_24',
    'roll_mean_168',
    'hour_sin',
    'hour_cos'
]

corr = model_df[heatmap_cols].corr()

plt.figure(figsize=(12, 8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(heatmap_cols)), heatmap_cols, rotation=90)
plt.yticks(range(len(heatmap_cols)), heatmap_cols)
plt.title('Correlation heatmap of key features')
plt.tight_layout()
plt.show()

In [ ]:
summer_profile = summer_df.groupby('hour')['Global_active_power'].mean()
winter_profile = winter_df.groupby('hour')['Global_active_power'].mean()

plt.figure(figsize=(10, 5))
plt.plot(summer_profile.index, summer_profile.values, marker='o', label='Summer')
plt.plot(winter_profile.index, winter_profile.values, marker='o', label='Winter')
plt.title('Average daily load profile: Summer vs Winter')
plt.xlabel('Hour of day')
plt.ylabel('Average Global Active Power (kW)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
summer_df.boxplot(column='Global_active_power', by='hour', grid=False)
plt.title('Summer: Load distribution by hour of day')
plt.suptitle('')
plt.xlabel('Hour of day')
plt.ylabel('Global Active Power (kW)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
winter_df.boxplot(column='Global_active_power', by='hour', grid=False)
plt.title('Winter: Load distribution by hour of day')
plt.suptitle('')
plt.xlabel('Hour of day')
plt.ylabel('Global Active Power (kW)')
plt.show()

In [ ]:
# each row is one 24-hour forecast, columns are hour-ahead 1..24
summer_abs_error = np.abs(summer_preds_kw - summer_trues_kw)

summer_hour_error = pd.DataFrame({
    'hour_ahead': np.arange(1, 25),
    'MAE': summer_abs_error.mean(axis=0),
    'RMSE': np.sqrt(((summer_preds_kw - summer_trues_kw) ** 2).mean(axis=0))
})

print(summer_hour_error)

plt.figure(figsize=(10, 5))
plt.plot(summer_hour_error['hour_ahead'], summer_hour_error['MAE'], marker='o', label='MAE')
plt.plot(summer_hour_error['hour_ahead'], summer_hour_error['RMSE'], marker='o', label='RMSE')
plt.title('Summer-only model: Error by forecast horizon')
plt.xlabel('Hour ahead')
plt.ylabel('Error')
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
winter_abs_error = np.abs(winter_preds_kw - winter_trues_kw)

winter_hour_error = pd.DataFrame({
    'hour_ahead': np.arange(1, 25),
    'MAE': winter_abs_error.mean(axis=0),
    'RMSE': np.sqrt(((winter_preds_kw - winter_trues_kw) ** 2).mean(axis=0))
})

print(winter_hour_error)

plt.figure(figsize=(10, 5))
plt.plot(winter_hour_error['hour_ahead'], winter_hour_error['MAE'], marker='o', label='MAE')
plt.plot(winter_hour_error['hour_ahead'], winter_hour_error['RMSE'], marker='o', label='RMSE')
plt.title('Winter-only model: Error by forecast horizon')
plt.xlabel('Hour ahead')
plt.ylabel('Error')
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
monthly_profile = model_df.copy()
monthly_profile['year_month'] = monthly_profile['datetime'].dt.to_period('M').astype(str)

monthly_avg = monthly_profile.groupby('year_month')['Global_active_power'].mean().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly_avg['year_month'], monthly_avg['Global_active_power'], marker='o')
plt.title('Monthly average Global Active Power')
plt.xlabel('Month')
plt.ylabel('Average Global Active Power (kW)')
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
trend_df = model_df[['datetime', 'Global_active_power']].copy()

trend_df['rolling_7d'] = trend_df['Global_active_power'].rolling(24 * 7).mean()
trend_df['rolling_30d'] = trend_df['Global_active_power'].rolling(24 * 30).mean()

plt.figure(figsize=(14, 5))
plt.plot(trend_df['datetime'], trend_df['Global_active_power'], alpha=0.25, label='Hourly load')
plt.plot(trend_df['datetime'], trend_df['rolling_7d'], label='7-day rolling mean')
plt.plot(trend_df['datetime'], trend_df['rolling_30d'], label='30-day rolling mean')
plt.title('Global Active Power with rolling mean trends')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
# ... your plotting code ...
plt.tight_layout()
plt.savefig('/content/figure_name.png', dpi=300, bbox_inches='tight')
plt.show()

## Visualizations

This section presents graphical analysis of the forecasting results. The plots are used to evaluate model fit, inspect forecast errors, compare summer and winter behavior, and illustrate the long-term seasonal structure of household electricity demand. Overall, the visual analysis shows that the summer-only LSTM model gives the best fit in terms of RMSE and MAE, while winter remains more difficult to predict because it has higher load and greater variability. The plots also confirm that the model captures the general daily and seasonal structure of electricity usage, although it tends to smooth sharp peaks.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(summer_true_flat[:500], label='Actual')
plt.plot(summer_pred_flat[:500], label='Predicted')
plt.title('Summer-only model: Actual vs Predicted')
plt.xlabel('Forecasted hourly points')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/actual_vs_predicted_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(summer_residuals, bins=50)
plt.title('Summer-only model: Residual distribution')
plt.xlabel('Residual (kW)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('/content/residual_histogram_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(summer_profile.index, summer_profile.values, marker='o', label='Summer')
plt.plot(winter_profile.index, winter_profile.values, marker='o', label='Winter')
plt.title('Average daily load profile: Summer vs Winter')
plt.xlabel('Hour of day')
plt.ylabel('Average Global Active Power (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/daily_profile_summer_vs_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
summer_true_flat = summer_trues_kw.flatten()
summer_pred_flat = summer_preds_kw.flatten()

plt.figure(figsize=(14, 5))
plt.plot(summer_true_flat[:500], label='Actual')
plt.plot(summer_pred_flat[:500], label='Predicted')
plt.title('Summer-only model: Actual vs Predicted load (first 500 forecast points)')
plt.xlabel('Forecasted hourly points')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/actual_vs_predicted_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
summer_true_flat = summer_trues_kw.flatten()
summer_pred_flat = summer_preds_kw.flatten()

plt.figure(figsize=(14, 5))
plt.plot(summer_true_flat[:500], label='Actual')
plt.plot(summer_pred_flat[:500], label='Predicted')
plt.title('Summer-only model: Actual vs Predicted load (first 500 forecast points)')
plt.xlabel('Forecasted hourly points')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/actual_vs_predicted_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(summer_true_flat[:2000], summer_pred_flat[:2000], alpha=0.4)
plt.xlabel('Actual load (kW)')
plt.ylabel('Predicted load (kW)')
plt.title('Summer-only model: Predicted vs Actual scatter')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/predicted_vs_actual_scatter_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(summer_residuals, bins=50)
plt.title('Summer-only model: Distribution of residuals')
plt.xlabel('Residual (kW)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/residual_histogram_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sample_idx = 100

plt.figure(figsize=(10, 5))
plt.plot(range(1, 25), summer_trues_kw[sample_idx], marker='o', label='Actual')
plt.plot(range(1, 25), summer_preds_kw[sample_idx], marker='o', label='Predicted')
plt.title(f'Summer-only model: 24-hour forecast example (sample {sample_idx})')
plt.xlabel('Hour ahead')
plt.ylabel('Global Active Power (kW)')
plt.xticks(range(1, 25, 2))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/forecast_24h_example_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
summer_abs_error = np.abs(summer_preds_kw - summer_trues_kw)

summer_hour_error = pd.DataFrame({
    'hour_ahead': np.arange(1, 25),
    'MAE': summer_abs_error.mean(axis=0),
    'RMSE': np.sqrt(((summer_preds_kw - summer_trues_kw) ** 2).mean(axis=0))
})

plt.figure(figsize=(10, 5))
plt.plot(summer_hour_error['hour_ahead'], summer_hour_error['MAE'], marker='o', label='MAE')
plt.plot(summer_hour_error['hour_ahead'], summer_hour_error['RMSE'], marker='o', label='RMSE')
plt.title('Summer-only model: Error by forecast horizon')
plt.xlabel('Hour ahead')
plt.ylabel('Error')
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/error_by_horizon_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
winter_abs_error = np.abs(winter_preds_kw - winter_trues_kw)

winter_hour_error = pd.DataFrame({
    'hour_ahead': np.arange(1, 25),
    'MAE': winter_abs_error.mean(axis=0),
    'RMSE': np.sqrt(((winter_preds_kw - winter_trues_kw) ** 2).mean(axis=0))
})

plt.figure(figsize=(10, 5))
plt.plot(winter_hour_error['hour_ahead'], winter_hour_error['MAE'], marker='o', label='MAE')
plt.plot(winter_hour_error['hour_ahead'], winter_hour_error['RMSE'], marker='o', label='RMSE')
plt.title('Winter-only model: Error by forecast horizon')
plt.xlabel('Hour ahead')
plt.ylabel('Error')
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/error_by_horizon_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
season_compare = pd.DataFrame({
    'season': ['summer', 'winter'],
    'RMSE': [summer_rmse, winter_rmse],
    'MAE': [summer_mae, winter_mae],
    'MAPE': [summer_mape, winter_mape]
})

print(season_compare)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['RMSE'])
plt.title('RMSE comparison: Summer vs Winter')
plt.ylabel('RMSE')
plt.tight_layout()
plt.savefig('/content/rmse_summer_vs_winter.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['MAE'])
plt.title('MAE comparison: Summer vs Winter')
plt.ylabel('MAE')
plt.tight_layout()
plt.savefig('/content/mae_summer_vs_winter.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(season_compare['season'], season_compare['MAPE'])
plt.title('MAPE comparison: Summer vs Winter')
plt.ylabel('MAPE (%)')
plt.tight_layout()
plt.savefig('/content/mape_summer_vs_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(summer_train_losses, marker='o', label='Train Loss')
plt.plot(summer_val_losses, marker='o', label='Validation Loss')
plt.title('Summer-only model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(winter_train_losses, marker='o', label='Train Loss')
plt.plot(winter_val_losses, marker='o', label='Validation Loss')
plt.title('Winter-only model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
heatmap_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
    'target',
    'lag_1',
    'lag_24',
    'lag_168',
    'roll_mean_24',
    'roll_mean_168',
    'hour_sin',
    'hour_cos'
]

corr = model_df[heatmap_cols].corr()

plt.figure(figsize=(12, 8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(heatmap_cols)), heatmap_cols, rotation=90)
plt.yticks(range(len(heatmap_cols)), heatmap_cols)
plt.title('Correlation heatmap of key features')
plt.tight_layout()
plt.savefig('/content/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
summer_profile = summer_df.groupby('hour')['Global_active_power'].mean()
winter_profile = winter_df.groupby('hour')['Global_active_power'].mean()

plt.figure(figsize=(10, 5))
plt.plot(summer_profile.index, summer_profile.values, marker='o', label='Summer')
plt.plot(winter_profile.index, winter_profile.values, marker='o', label='Winter')
plt.title('Average daily load profile: Summer vs Winter')
plt.xlabel('Hour of day')
plt.ylabel('Average Global Active Power (kW)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/daily_profile_summer_vs_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
summer_df.boxplot(column='Global_active_power', by='hour', grid=False)
plt.title('Summer: Load distribution by hour of day')
plt.suptitle('')
plt.xlabel('Hour of day')
plt.ylabel('Global Active Power (kW)')
plt.tight_layout()
plt.savefig('/content/boxplot_summer_by_hour.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 5))
winter_df.boxplot(column='Global_active_power', by='hour', grid=False)
plt.title('Winter: Load distribution by hour of day')
plt.suptitle('')
plt.xlabel('Hour of day')
plt.ylabel('Global Active Power (kW)')
plt.tight_layout()
plt.savefig('/content/boxplot_winter_by_hour.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
monthly_profile = model_df.copy()
monthly_profile['year_month'] = monthly_profile['datetime'].dt.to_period('M').astype(str)

monthly_avg = monthly_profile.groupby('year_month')['Global_active_power'].mean().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly_avg['year_month'], monthly_avg['Global_active_power'], marker='o')
plt.title('Monthly average Global Active Power')
plt.xlabel('Month')
plt.ylabel('Average Global Active Power (kW)')
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/monthly_average_load.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
trend_df = model_df[['datetime', 'Global_active_power']].copy()
trend_df['rolling_7d'] = trend_df['Global_active_power'].rolling(24 * 7).mean()
trend_df['rolling_30d'] = trend_df['Global_active_power'].rolling(24 * 30).mean()

plt.figure(figsize=(14, 5))
plt.plot(trend_df['datetime'], trend_df['Global_active_power'], alpha=0.25, label='Hourly load')
plt.plot(trend_df['datetime'], trend_df['rolling_7d'], label='7-day rolling mean')
plt.plot(trend_df['datetime'], trend_df['rolling_30d'], label='30-day rolling mean')
plt.title('Global Active Power with rolling mean trends')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/rolling_trend_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
visual_summary = pd.DataFrame({
    'Model': ['All-season improved', 'Summer-only', 'Winter-only'],
    'RMSE': [0.5938753250109605, 0.500829232063017, 0.7926112882648556],
    'MAE': [0.4206670947001056, 0.3356997745493183, 0.5819381981558279],
    'MAPE': [54.17787234086738, 60.47003695097042, 56.73434859161859]
})

print(visual_summary)
visual_summary.to_csv('/content/visual_summary_metrics.csv', index=False)

## Title
Short-Term Household Electricity Forecasting with LSTM and Seasonal Modeling

## Introduction

Accurate short-term electricity forecasting is important for energy planning, demand management, and efficient operation of power systems. Forecasting models help estimate future electricity consumption from historical usage patterns and related variables, which can support both households and utilities.

This project studies short-term household electricity forecasting using the Individual Household Electric Power Consumption dataset. The goal is to predict the next 24 hours of electricity demand from the previous 192 hours of hourly data. The work is inspired by the uploaded paper, which uses household electricity data, time-related features, dimensionality reduction, clustering, and neural forecasting models. In that paper, LSTM was the strongest neural baseline, and the best overall setup combined kernel PCA, k-means clustering with 4 clusters, and LSTM forecasting. The paper also analyzed forecasting performance by season and used a 192-hour input window to predict the next 24 hours.  

The dataset used in this notebook contains only one household, so the full clustering-based improvement strategy from the paper cannot be reproduced directly. Because of that, this project focuses on a valid adaptation of the paper’s ideas: LSTM-based forecasting, feature engineering, and seasonal modeling with separate summer and winter experiments.

## Problem Statement

The main task in this project is to build a short-term forecasting model that predicts the next 24 hours of household electricity consumption from the previous 192 hours of historical hourly data.

More specifically, the project aims to:

1. preprocess and clean the household electricity dataset,
2. transform minute-level observations into hourly observations,
3. engineer useful forecasting features such as cyclical calendar variables, lag variables, and rolling statistics,
4. train LSTM-based forecasting models,
5. compare all-season, summer-only, and winter-only forecasting performance,
6. evaluate results using RMSE, MAE, and MAPE,
7. compare the findings with the uploaded paper.

Because the available dataset contains only one household, the project does not fully implement the paper’s multi-household clustering pipeline. Instead, it tests whether seasonal LSTM modeling can improve forecasting performance on a single-household dataset.

## Dataset Description

The dataset used in this notebook is the Individual Household Electric Power Consumption dataset. It contains measurements of one household’s electric power usage over several years. The original data includes a date column, a time column, and multiple electrical variables such as global active power, global reactive power, voltage, current intensity, and three sub-metering channels.

The original dataset was provided at minute resolution. Since the forecasting setup in the paper is based on hourly demand prediction, the minute-level data was resampled into hourly averages before modeling. Missing values were handled through interpolation so that the time series would remain continuous.

The main target variable in this notebook is **Global_active_power**, which represents household active power consumption. Additional variables such as reactive power, voltage, current intensity, and sub-metering channels were used as explanatory inputs.

## Methodology

The methodology used in this notebook follows the general forecasting structure described in the uploaded paper, with modifications to fit a single-household dataset.

### 1. Data preprocessing
The original dataset was loaded, cleaned, and converted into a proper datetime format by merging the separate date and time columns. Missing values in the numerical variables were filled using interpolation. The minute-level time series was then resampled into hourly data.

### 2. Target transformation
The target variable, Global_active_power, was transformed using a logarithmic transformation, `log1p(x)`, to stabilize variance and reduce the influence of extreme values. The paper also applies a logarithmic transformation as part of preprocessing. :contentReference[oaicite:2]{index=2}

### 3. Feature engineering
Several groups of input features were created:
- calendar features derived from the timestamp,
- cyclical encoding of hour, day, and month using sine and cosine transformations,
- lag features such as 1-hour, 24-hour, 48-hour, and 168-hour lags,
- rolling mean and rolling standard deviation features,
- original electrical variables such as reactive power, voltage, current intensity, and sub-metering values.

This design follows the paper’s idea of combining historical load information with time-related explanatory variables. :contentReference[oaicite:3]{index=3}

### 4. Seasonal modeling
The uploaded paper evaluates forecasting separately for summer and winter because consumption behavior differs by season. Following that idea, this notebook created separate seasonal subsets and trained dedicated summer-only and winter-only models in addition to an all-season model.

### 5. Sequence construction
The supervised learning samples were created using a sliding-window method:
- input window = previous 192 hours,
- forecast horizon = next 24 hours.

This directly matches the forecasting setup reported in the paper.

### 6. Model training
The main forecasting model used in this notebook is an LSTM network implemented in PyTorch. LSTM was chosen because the paper reports that it performed best among the tested neural forecasting models. Models were trained with early stopping, dropout, and validation monitoring to reduce overfitting.

### 7. Evaluation
Forecast performance was evaluated using:
- Root Mean Squared Error (RMSE),
- Mean Absolute Error (MAE),
- Mean Absolute Percentage Error (MAPE).

These metrics allow comparison between the different seasonal models and connect the results back to the paper’s evaluation framework.

## Experimental Setup

The dataset was divided into train, validation, and test sets using chronological order, so that older data was used for training and newer data was used for validation and testing. This mirrors a realistic forecasting scenario, where future values must be predicted from past observations.

Three main experiments were carried out:

1. **All-season LSTM model**  
   A single LSTM was trained on the full dataset after preprocessing and feature engineering.

2. **Summer-only LSTM model**  
   The data was filtered to summer months, and a dedicated LSTM was trained only on summer observations.

3. **Winter-only LSTM model**  
   The data was filtered to winter months, and a dedicated LSTM was trained only on winter observations.

Additional winter experiments were also tested, including an extra derived feature and a larger LSTM architecture, to determine whether winter forecasting could be improved.

## Notebook Roadmap

The rest of this notebook is organized as follows:

1. data loading and cleaning,
2. hourly resampling and preprocessing,
3. feature engineering,
4. sequence generation,
5. LSTM model construction,
6. seasonal experiments,
7. evaluation metrics,
8. visual analysis,
9. discussion and comparison with the paper,
10. conclusion.

## Methodology Summary

This notebook uses hourly household electricity data to predict the next 24 hours from the previous 192 hours. After cleaning and interpolation, the data is transformed with logarithmic scaling, enriched with time-based and lag-based features, and used to train LSTM forecasting models. Separate all-season, summer-only, and winter-only experiments are performed to study the effect of seasonal demand patterns.

## Discussion

This project applied LSTM-based short-term load forecasting to the Individual Household Electric Power Consumption dataset. The main objective was to predict the next 24 hours of electricity usage from the previous 192 hours of hourly data, following the forecasting setup used in the paper. The workflow included data cleaning, hourly resampling, interpolation of missing values, logarithmic transformation of the target, calendar-based feature engineering, lag features, rolling statistics, and seasonal model comparison.

The results showed that seasonal modeling was useful. The summer-only model achieved the best RMSE and MAE, while the winter-only model was consistently harder to predict. This difference was also visible in the visualizations: winter had higher average demand, greater spread, and stronger evening peaks than summer. These findings support the idea that electricity demand patterns differ substantially by season and that using separate seasonal models can improve forecasting performance.

The summer-only LSTM produced the strongest overall error results in terms of RMSE and MAE. The actual-versus-predicted plots showed that the model successfully captured the broad daily usage pattern, but it often smoothed sharp peaks and underpredicted sudden spikes. The residual plots showed that errors were centered near zero overall, but with a longer negative tail, indicating occasional underprediction. The loss curves showed that validation loss stopped improving after a small number of epochs, which justified the use of early stopping.

The winter-only model produced larger errors than the summer-only model. The daily profile and boxplots suggested one likely reason: winter demand was not only higher, but also more variable and more extreme, especially in the evening. This made winter forecasting harder. Attempts to improve the winter model by adding an unmetered-load feature or increasing LSTM hidden size did not produce a clear overall improvement, which suggests that the main difficulty is the underlying complexity of winter demand rather than only model size.

A key limitation of this work is that the dataset contains only one household. The paper’s strongest improvement comes from applying dimensionality reduction and clustering to a large set of households before training separate models per cluster. That approach cannot be reproduced directly on this single-household dataset. For that reason, the most valid adaptation here was to emphasize seasonal modeling rather than clustering. This means the present work is only partially comparable to the paper: it reproduces the forecasting horizon, the LSTM baseline idea, and the seasonal structure, but not the full multi-household clustering pipeline.

Another important observation is that RMSE and MAE were more stable and interpretable than MAPE in this dataset. Because the load can be relatively low during many hours, percentage-based error can become inflated. This is why the report should place more weight on RMSE and MAE when judging model quality, while still reporting MAPE for consistency with the paper.

Overall, the experiments support three main conclusions. First, LSTM is a strong baseline for short-term household electricity forecasting. Second, seasonal structure matters and can improve performance when modeled explicitly. Third, the largest gains reported in the paper likely depend on access to many households so that clustering can identify groups with similar consumption behavior.\\

## Conclusion:

This project developed and evaluated LSTM-based models for short-term household electricity forecasting using hourly power consumption data. The forecasting task used the previous 192 hours of input data to predict the next 24 hours, matching the setup described in the paper.

Among the tested models, the summer-only LSTM achieved the best RMSE and MAE, while the all-season model produced the best MAPE. Winter forecasting was the most difficult case, with consistently higher errors than summer. Visual analysis confirmed that the model captured the overall daily and seasonal structure of electricity demand, but it often smoothed sharp peaks and underpredicted sudden high-load events.

The experiments support the paper’s general conclusion that LSTM is a strong method for electricity load forecasting and that seasonal structure is important. However, the full improvement strategy proposed in the paper could not be reproduced because the dataset used here contains only one household, while the paper’s strongest results depend on clustering many households after dimensionality reduction.

For this dataset, the most effective practical strategy was seasonal modeling rather than clustering. The best overall model in terms of RMSE and MAE was the summer-only LSTM. Future work should test the same pipeline on a true multi-household dataset so that dimensionality reduction and clustering can be applied in the same way as in the paper. Additional future improvements could include testing GRU, Transformer-based forecasting models, richer weather variables, and attention-based sequence models.\\

##Final Summary

The notebook showed that seasonal LSTM forecasting is effective for one-household electricity data. Summer forecasts were more accurate than winter forecasts, and visual analysis explained this by showing that winter demand is higher and more variable. While the full clustering-based improvement from the paper could not be reproduced on a single-household dataset, the experiments still confirmed that LSTM and seasonal modeling are strong tools for short-term electricity forecasting.

In [ ]:
final_results.to_csv('/content/final_results.csv', index=False)
print('Saved to /content/final_results.csv')

In [ ]:
import shutil

shutil.copy(
    "/content/drive/MyDrive/myfile.txt",
    "/content/YOUR_REPO/myfile.txt"
)